In [1]:
import pandas as pd

In [2]:
DATA_FILE = "IMDB Dataset.csv"

In [3]:
df = pd.read_csv(DATA_FILE)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

print(f"Train set: {train_df.shape}")
print(f"Validation set: {val_df.shape}")
print(f"Test set: {test_df.shape}")

Train set: (28000, 2)
Validation set: (7000, 2)
Test set: (15000, 2)


In [10]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stopwords_list = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower()
    words = nltk.word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stopwords_list]
    return " ".join(words)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pagarwa1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pagarwa1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\pagarwa1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
train_df["review"].loc[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [ ]:
clean_text(train_df["review"].loc[0])

'one reviewer mentioned watching oz episode hooked right exactly happened first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda em city home many aryan muslim gangsta latino christian italian irish scuffle death stare dodgy dealing shady agreement never far away would say main appeal show due fact go show dare forget pretty picture painted mainstream audience forget charm forget romance oz mess around first episode ever saw struck nasty surreal say ready watched developed taste oz got accustomed high level graphic violence violence injustice crooked guard sold nickel inmate kill order get away well mannered middle class inmate turned prison bitch due lack street skill prison experience watching oz m

In [13]:
train_df["review_cleaned"] = train_df["review"].apply(clean_text)
val_df["review_cleaned"] = val_df["review"].apply(clean_text)
test_df["review_cleaned"] = test_df["review"].apply(clean_text)

In [14]:
# Bag of Words

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000)
X_train_bow = vectorizer.fit_transform(train_df["review_cleaned"]).toarray()
X_val_bow = vectorizer.transform(val_df["review_cleaned"]).toarray()
X_test_bow = vectorizer.transform(test_df["review_cleaned"]).toarray()

In [15]:
y_train = train_df["sentiment"].map({"positive": 1, "negative": 0}).values
y_val = val_df["sentiment"].map({"positive": 1, "negative": 0}).values
y_test = test_df["sentiment"].map({"positive": 1, "negative": 0}).values

In [16]:
print(X_train_bow.shape, y_train.shape)
print(X_val_bow.shape, y_val.shape)
print(X_test_bow.shape, y_test.shape)

(28000, 5000) (28000,)
(7000, 5000) (7000,)
(15000, 5000) (15000,)


In [17]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier


rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_bow, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
y_val_pred = rf_classifier.predict(X_val_bow)

In [19]:
from sklearn.metrics import accuracy_score, classification_report


accuracy_score(y_val, y_val_pred)

0.8487142857142858

In [21]:
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

           0       0.84      0.86      0.85      3540
           1       0.86      0.83      0.85      3460

    accuracy                           0.85      7000
   macro avg       0.85      0.85      0.85      7000
weighted avg       0.85      0.85      0.85      7000



In [24]:
# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV


param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_classifier,
    param_distributions=param_dist,
    n_iter=3,
    cv=2,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
rf_random_search.fit(X_train_bow, y_train)

Fitting 2 folds for each of 3 candidates, totalling 6 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,n_iter,3
,scoring,None
,n_jobs,-1
,refit,True
,cv,2
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan
